## 🛠️ 실습 4 - Tool 전략과 Planning 비교 실험

**주제:** 최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례 조사

> 목표: 같은 문제를 **단일 Tool**, **Tool 조합**, **Planning 기반 접근**으로 각각 해결해 보고  
> 결과의 **정보 품질 / 구조 / 안정성** 차이를 비교합니다.

## 🛠️ 프로젝트 개발 환경 구축

이 노트북은 **CrewAI**와 **OpenAI**를 활용한 에이전트 개발 환경을 전제로 합니다.  
아래 단계에 따라 가상환경을 만들고 필요한 패키지를 설치해 주세요.

### 1. 가상환경 설정
터미널(Terminal)에서 실행합니다.

* **가상환경 생성:** `env04`라는 독립된 환경을 구축합니다.
    ```bash
    python -m venv env04
    ```

* **가상환경 활성화:** 생성된 환경으로 진입합니다. (Windows CMD 기준)
    ```bash
    env04\Scripts\activate
    ```

### 2. 필수 패키지 설치
필요한 패키지를 설치합니다.

```bash
pip install crewai crewai-tools openai python-dotenv exa-py
```

### 3. API 키 준비
프로젝트 루트에 `.env` 파일을 만들고 아래 값을 설정하세요.

```env
OPENAI_API_KEY=your_openai_api_key
OPENAI_MODEL=gpt-4o-mini
SERPER_API_KEY=your_serper_api_key
EXA_API_KEY=your_exa_api_key
```

> 실습 포인트  
> - **실험 1:** 단일 Tool 접근  
> - **실험 2:** Multi-Tool 접근  
> - **실험 3:** Planning-first 접근


### 1️⃣ 환경 변수 로드

In [1]:
# .env 파일에 아래 키가 정의되어 있어야 합니다.
# - OPENAI_API_KEY
# - OPENAI_MODEL (예: gpt-4o-mini 등)  ※ 선택
# - SERPER_API_KEY
# - EXA_API_KEY

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

assert SERPER_API_KEY, "SERPER_API_KEY가 .env에 없습니다."
assert EXA_API_KEY, "EXA_API_KEY가 .env에 없습니다."

print("✅ 환경 변수 로드 완료:", {"OPENAI_MODEL": OPENAI_MODEL})


AssertionError: EXA_API_KEY가 .env에 없습니다.

### 2️⃣ Tool 연결 (SERPER / EXA)

In [18]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool
from crewai.tools import tool
from exa_py import Exa

serper_tool = SerperDevTool(api_key=SERPER_API_KEY)  # SERPER_API_KEY를 명시적으로 전달하여 어떤 키가 사용되는지 명확히 합니다.
exa = Exa(api_key=EXA_API_KEY) # Exa API를 활용한 의미 기반 검색 도구입니다.

@tool("exa_semantic_search") # 이 함수는 EXA API를 활용하여 의미 기반 검색을 수행하는 도구임. Agent가 이 도구를 호출할 때, 사용자가 입력한 쿼리를 기반으로 EXA API를 통해 관련된 기업 사례를 검색하여 반환함.
def exa_semantic_search(query: str) -> str:
    """의미 기반 검색으로 기업 사례를 확장합니다."""   
    improved_query = f"""
    Find real-world company cases about AI-based data analysis automation.
    Focus on cases where organizations used AI / ML / analytics automation to improve
    business analysis, forecasting, reporting, decision-making, anomaly detection, or process efficiency.
    Include company name, business problem, applied AI/data technology, and measurable outcomes if available.
    Prefer credible sources such as company blogs, case studies, reputable media, analyst reports, or research summaries.

    User query: {query}
    """.strip()

    result = exa.search_and_contents( # EXA API를 활용하여 의미 기반 검색을 수행하는 부분입니다. 사용자가 입력한 쿼리를 기반으로 EXA API를 통해 관련된 기업 사례를 검색하여 반환합니다.
        improved_query,
        type="neural", # 의미 기반 검색을 수행하기 위해 "neural" 타입을 지정합니다. 이는 EXA API가 의미 기반 검색을 수행하도록 지시하는 역할을 합니다.
        num_results=5, # 검색 결과로 최대 5개의 관련 사례를 반환하도록 지정합니다. 이는 사용자가 너무 많은 정보를 받지 않도록 하면서도 충분한 사례를 제공하기 위한 설정입니다.
        highlights=True, # 검색 결과에서 중요한 부분을 강조 표시하여 반환하도록 지정합니다. 이를 통해 사용자가 검색 결과에서 핵심 정보를 빠르게 파악할 수 있도록 도와줍니다.
        text=True, # 검색 결과의 전체 텍스트를 반환하도록 지정합니다. 이는 사용자가 검색 결과의 세부 내용을 확인할 수 있도록 하는 설정입니다.
    )

    items = []
    for r in result.results:
        title = getattr(r, "title", "") or "(no title)" # 검색 결과에서 제목을 가져옵니다. 제목이 없는 경우 "(no title)"로 표시합니다.
        url = getattr(r, "url", "") or "" # 검색 결과에서 URL을 가져옵니다. URL이 없는 경우 빈 문자열로 표시합니다.
        summary = "" # 
        if getattr(r, "highlights", None): # 검색 결과에서 강조 표시된 부분이 있는 경우, 이를 요약으로 사용합니다. 최대 3개의 강조 표시를 " | "로 구분하여 하나의 문자열로 만듭니다.
            summary = " | ".join(r.highlights[:3]) 
        elif getattr(r, "text", None): # 검색 결과에서 전체 텍스트가 있는 경우, 이를 요약으로 사용합니다. 최대 500자까지 가져옵니다.
            summary = r.text[:500] 
        items.append(f"- 제목: {title}\n  URL: {url}\n  요약: {summary}") # 검색 결과를 제목, URL, 요약으로 구성된 문자열로 포맷하여 items 리스트에 추가합니다.

    return "\n\n".join(items) if items else "검색 결과가 없습니다."


llm = LLM(model=OPENAI_MODEL)

print("✅ SERPER / EXA Tool 연결 완료")


✅ SERPER / EXA Tool 연결 완료


### 3️⃣ 실습 미션 설정

In [19]:
mission = "최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라"

required_fields = [
    "기업명",
    "산업",
    "도입 배경",
    "실행 방식",
    "성과/효과",
    "출처 URL"
]

print("✅ 실습 미션:", mission)
print("✅ 정리 항목:", required_fields)


✅ 실습 미션: 최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라
✅ 정리 항목: ['기업명', '산업', '도입 배경', '실행 방식', '성과/효과', '출처 URL']


### 4️⃣ 공통 출력 형식 정의

> 세 실험에서 공통으로 사용하는 출력 형식을 미리 정의합니다.  
> 동일한 형식을 사용해야 실험 간 결과 비교가 의미 있습니다.


In [20]:
output_format = """
[출력 형식]
- 총 4~6개 사례를 정리하라.
- 국내 1개 이상 + 해외 2개 이상 포함하라.
- 아래 항목을 각 사례마다 포함하라.
  1) 기업명
  2) 산업
  3) 도입 배경
  4) 실행 방식
  5) 성과/효과
  6) 출처 URL
- 마지막에 전체 비교 요약 5줄 이내로 정리하라.
- 가능한 경우 표 또는 번호 목록 형태로 구조화하라.
""".strip()

print(output_format)


[출력 형식]
- 총 4~6개 사례를 정리하라.
- 국내 1개 이상 + 해외 2개 이상 포함하라.
- 아래 항목을 각 사례마다 포함하라.
  1) 기업명
  2) 산업
  3) 도입 배경
  4) 실행 방식
  5) 성과/효과
  6) 출처 URL
- 마지막에 전체 비교 요약 5줄 이내로 정리하라.
- 가능한 경우 표 또는 번호 목록 형태로 구조화하라.


### 5️⃣ 실험 1: 단일 Tool Agent 정의 (SERPER만 사용)

In [21]:
single_tool_agent = Agent(
    role="Research Analyst (Single Tool)",
    goal="단일 검색 Tool만 사용해 기업 사례를 빠르게 조사한다.",
    backstory="당신은 빠르게 자료를 찾는 실무형 리서처다. 단일 검색 Tool 환경에서 근거 중심으로 사례를 정리한다. 모든 결과물은 반드시 한국어로 작성한다.",
    tools=[serper_tool],
    llm=llm,
    max_iter=5,  # Tool 호출 횟수 제한
    verbose=True
)

single_tool_task = Task(
    description=f"""
    [미션]
    {mission}

    [접근 방식]
    - SERPER 검색 Tool 하나만 사용하라.
    - 너무 길게 탐색하지 말고 빠르게 핵심 사례를 정리하라.

    {output_format}
    """.strip(),
    expected_output="단일 Tool만 사용한 기업 사례 조사 결과",
    agent=single_tool_agent
)

print("✅ 실험 1 준비 완료")


✅ 실험 1 준비 완료


### 6️⃣ 실험 2: Tool 조합 Agent 정의 (SERPER + EXA)

In [22]:
multi_tool_agent = Agent(
    role="Research Analyst (Multi Tool)",
    goal="여러 검색 Tool을 조합해 더 풍부한 기업 사례를 조사한다.",
    backstory="당신은 서로 다른 검색 방식을 조합해 더 깊고 다양한 사례를 확보하는 리서치 전문가다. 모든 결과물은 반드시 한국어로 작성한다.",
    tools=[serper_tool, exa_semantic_search],
    llm=llm,
    max_iter=7,  # Tool 호출 횟수 제한 (Multi-Tool이므로 단일보다 여유 있게 설정)
    verbose=True
)

multi_tool_task = Task(
    description=f"""
    [미션]
    {mission}

    [접근 방식]
    - SERPER와 EXA semantic search를 조합하라.
    - 서로 다른 검색 결과를 비교·보완하여 사례를 확장하라.

    {output_format}
    """.strip(),
    expected_output="여러 Tool을 조합한 기업 사례 조사 결과",
    agent=multi_tool_agent
)

print("✅ 실험 2 준비 완료")


✅ 실험 2 준비 완료


### 7️⃣ 실험 3: Planning-first Agent 정의

> `reasoning=True`: Agent가 실행 전에 내부적으로 계획을 세우도록 활성화하는 옵션입니다.  
> 이를 통해 Agent는 바로 Tool을 호출하지 않고, 먼저 작업 순서를 구성한 뒤 실행합니다.


In [23]:
planning_agent = Agent(
    role="Planning-first Research Strategist",
    goal="실행 전에 계획을 세우고, 계획에 따라 기업 사례를 구조적으로 조사한다.",
    backstory=(
        "당신은 복잡한 조사 과제를 바로 실행하지 않고 먼저 계획을 세운 뒤 "
        "검색과 정리를 수행하는 전략형 에이전트다. 모든 결과물은 반드시 한국어로 작성한다."
    ),
    tools=[serper_tool, exa_semantic_search],
    llm=llm,
    max_iter=7,  # Tool 호출 횟수 제한 (Planning-first이므로 여유 있게 설정)
    reasoning=True,  # 실행 전에 내부적으로 계획을 세우도록 활성화하는 옵션입니다.
    verbose=True
)

planning_task = Task(
    description=f"""
    [미션]
    {mission}

    [중요]
    아래 구조를 반드시 따르라.

    1) Plan
    - 조사에 필요한 작업 단계를 먼저 3~5줄로 계획하라.
    - 어떤 Tool을 어느 단계에서 사용할지 간단히 밝히라.

    2) Execute
    - 계획에 따라 실제 사례를 조사하라.
    - 최종 사례 정리는 아래 출력 형식을 따르라.

    3) Reflect
    - 결과를 스스로 점검하라.
    - 사례 중복, 출처 부족, 국내/해외 균형 부족 여부를 3~5줄로 검토하라.

    {output_format}
    """.strip(),
    expected_output="Planning-first 방식의 기업 사례 조사 결과",
    agent=planning_agent
)

print("✅ 실험 3 준비 완료")


✅ 실험 3 준비 완료


### 8️⃣ Crew 구성 (단일 Tool)

In [24]:
crew_single = Crew(
    agents=[single_tool_agent],
    tasks=[single_tool_task],
    verbose=True
)

### 8️⃣-B Crew 실행과 최종 결과 출력 (단일 Tool)


In [25]:
single_result = crew_single.kickoff()

print("\n===== 단일 Tool 결과 =====\n")
print(single_result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.11.0                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: db95e1fc-b231-4fa3-abed-f9c96e1ab3a6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER 검색 Tool 하나만 사용하라.                                                                        │
│      - 너무 길게 탐색하지 말고 빠르게 핵심 사례를 정리하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│  ID: c85ea0e2-b021-4cdb-8dc7-ee9d2bcf1aa3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Single Tool)                                                                          │
│                                                                                                                 │
│  Task: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER 검색 Tool 하나만 사용하라.                                                                        │
│      - 너무 길게 탐색하지 말고 빠르게 핵심 사례를 정리하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AI 기반 데이터 분석 자동화 도입 사례 2021 2022 2023'}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Single Tool)                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | 번호 | 기업명                | 산업         | 도입 배경                                            | 실행    │
│  방식                                               | 성과/효과                                          |      │
│  출처 URL                                                                                       |               │
│  |------|--------------------|-------------|---------------------------------------------------|--------------  │
│  ---------------------------------------|-------------------------------------------------|-------------------  │
│  --------------------------------------------------------------------------|                                    │
│  | 1    | 삼성 SDS           | IT/서비스     | 데이터 분석 효율성을 높이기 위해 AI 기술 도입 필요성 인식        │
│  | AI 기반 분석 플랫폼 구축, 고객 데이터 분석                          | 고객 맞춤형 서비스 제공 및 판매량      │
│  증가                      | [출처](https://www.samsungsds.com/kr/insights/ai_use_cases.html)                   │
│  |                                                                                                              │
│  | 2    | 월마트             | 유통         | 고객 경험 개선 및 운영 비용 절감 필요성                           │
│  | '내 도우미' 앱을 통해 데이터 분석 및 고객 서비스 자동화                 | 직원 효율성 증가 및 고객 서비스    │
│  개선                     | [출처](https://blog.dfinite.ai/ai-adoption-case-studies)                            │
│  |                                                                                                              │
│  | 3    | 모건스탠리          | 금융         | 데이터 기반 의사결정 필요성 증대                                 │
│  | AI를 활용한 고객 포트폴리오 관리 및 시장 분석                           | 투자 성과 향상 및 리스크 관리      │
│  개선                      |                                                                                    │
│  [출처](https://www.deloitte.com/kr/ko/issues/generative-ai/state-of-ai-in-enterprise.html) |                   │
│  | 4    | LG 전자            | 전자/가전     | 제조 공정 최적화 및 품질 개선 필요                               │
│  | AI 기반 제조 설비 모니터링 및 결함 감지 시스템 도입                     | 생산성 향상 및 불량률 감소         │
│  | [출처](https://kita.net/researchTrade/report/tradeVoice/downloadTradeVoiceFile.do)         |                 │
│  | 5    | 카카오             | IT/서비스     | 사용자 데이터 분석을 통한 서비스 개인화 필요성 증대              │
│  | AI 기반 추천 시스템 및 고객 데이터 분석 도입                             | 사용자 경험 개선 및 앱 사용 증가  │
│  | [출처](http://www.kisdi.re.kr/report/fileView.do?key=m2102058837181&arrMasterId=4334696&id=1880176) |        │
│  | 6    | Amazon             | 유통         | 고객 맞춤형 서비스 제공 필요                                      │
│  | AI 를 이용한 추천 시스템 및 물류 관리 최적화                              | 판매량 증대 및 고객 충성도 향상  │
│  | [출처](https://www.alchera.ai/resource/blog/enterprise-ai-use-cases)                       |                 │
│                                                                                                                 │
│  **비교 요약:**                                                                                                 │
│  AI 기반 데이터 분석 자동화 도입은 국내외 기업들의 운영 효율성을 극대화하고 고객 만족도를 높이는 데 중요한      │
│  역할을 하고 있다. 삼성 SDS와 LG 전자는 IT 및 전자 산업에서 데이터 분석을 통해 품질 개선의 성과를 보고한 반면,  │
│  월마트와 Amazon은 유통 분야에서 고객 경험 향상을 이루어냈다. 모건스탠리, 카카오는 금융 및 IT/서비스 산업에서   │
│  개인화된 서비스로 시장 경쟁력을 강화하고 있다.                                                                 │
│                                                                                                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER 검색 Tool 하나만 사용하라.                                                                        │
│      - 너무 길게 탐색하지 말고 빠르게 핵심 사례를 정리하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│  Agent: Research Analyst (Single Tool)                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== 단일 Tool 결과 =====

| 번호 | 기업명                | 산업         | 도입 배경                                            | 실행 방식                                               | 성과/효과                                          | 출처 URL                                                                                       |
|------|--------------------|-------------|---------------------------------------------------|-----------------------------------------------------|-------------------------------------------------|---------------------------------------------------------------------------------------------|
| 1    | 삼성 SDS           | IT/서비스     | 데이터 분석 효율성을 높이기 위해 AI 기술 도입 필요성 인식                 | AI 기반 분석 플랫폼 구축, 고객 데이터 분석                          | 고객 맞춤형 서비스 제공 및 판매량 증가                      | [출처](https://www.samsungsds.com/kr/insights/ai_use_cases.html)                           |
| 2    | 월마트             | 유통         | 고객 경험 개선 및 운영 비용 절감 필요성                                      | '내 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: db95e1fc-b231-4fa3-abed-f9c96e1ab3a6                                                                       │
│  Final Output: | 번호 | 기업명                | 산업         | 도입 배경                                        │
│  | 실행 방식                                               | 성과/효과                                          │
│  | 출처 URL                                                                                       |             │
│  |------|--------------------|-------------|---------------------------------------------------|--------------  │
│  ---------------------------------------|-------------------------------------------------|-------------------  │
│  --------------------------------------------------------------------------|                                    │
│  | 1    | 삼성 SDS           | IT/서비스     | 데이터 분석 효율성을 높이기 위해 AI 기술 도입 필요성 인식        │
│  | AI 기반 분석 플랫폼 구축, 고객 데이터 분석                          | 고객 맞춤형 서비스 제공 및 판매량      │
│  증가                      | [출처](https://www.samsungsds.com/kr/insights/ai_use_cases.html)                   │
│  |                                                                                                              │
│  | 2    | 월마트             | 유통         | 고객 경험 개선 및 운영 비용 절감 필요성                           │
│  | '내 도우미' 앱을 통해 데이터 분석 및 고객 서비스 자동화                 | 직원 효율성 증가 및 고객 서비스    │
│  개선                     | [출처](https://blog.dfinite.ai/ai-adoption-case-studies)                            │
│  |                                                                                                              │
│  | 3    | 모건스탠리          | 금융         | 데이터 기반 의사결정 필요성 증대                                 │
│  | AI를 활용한 고객 포트폴리오 관리 및 시장 분석                           | 투자 성과 향상 및 리스크 관리      │
│  개선                      |                                                                                    │
│  [출처](https://www.deloitte.com/kr/ko/issues/generative-ai/state-of-ai-in-enterprise.html) |                   │
│  | 4    | LG 전자            | 전자/가전     | 제조 공정 최적화 및 품질 개선 필요                               │
│  | AI 기반 제조 설비 모니터링 및 결함 감지 시스템 도입                     | 생산성 향상 및 불량률 감소         │
│  | [출처](https://kita.net/researchTrade/report/tradeVoice/downloadTradeVoiceFile.do)         |                 │
│  | 5    | 카카오             | IT/서비스     | 사용자 데이터 분석을 통한 서비스 개인화 필요성 증대              │
│  | AI 기반 추천 시스템 및 고객 데이터 분석 도입                             | 사용자 경험 개선 및 앱 사용 증가  │
│  | [출처](http://www.kisdi.re.kr/report/fileView.do?key=m2102058837181&arrMasterId=4334696&id=1880176) |        │
│  | 6    | Amazon             | 유통         | 고객 맞춤형 서비스 제공 필요                                      │
│  | AI 를 이용한 추천 시스템 및 물류 관리 최적화                              | 판매량 증대 및 고객 충성도 향상  │
│  | [출처](https://www.alchera.ai/resource/blog/enterprise-ai-use-cases)                       |                 │
│                                                                                                                 │
│  **비교 요약:**                                                                                                 │
│  AI 기반 데이터 분석 자동화 도입은 국내외 기업들의 운영 효율성을 극대화하고 고객 만족도를 높이는 데 중요한      │
│  역할을 하고 있다. 삼성 SDS와 LG 전자는 IT 및 전자 산업에서 데이터 분석을 통해 품질 개선의 성과를 보고한 반면,  │
│  월마트와 Amazon은 유통 분야에서 고객 경험 향상을 이루어냈다. 모건스탠리, 카카오는 금융 및 IT/서비스 산업에서   │
│  개인화된 서비스로 시장 경쟁력을 강화하고 있다.                                                                 │
│                                                                                                   

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 9️⃣ Crew 구성 (Tool 조합)

In [26]:
crew_multi = Crew(
    agents=[multi_tool_agent],
    tasks=[multi_tool_task],
    verbose=True
)

### 9️⃣-B Crew 실행과 최종 결과 출력 (Tool 조합)


In [27]:
multi_result = crew_multi.kickoff()

print("\n===== Tool 조합 결과 =====\n")
print(multi_result)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.11.0                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8a83149f-564a-4478-a654-a212cb5fac34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER와 EXA semantic search를 조합하라.                                                                 │
│      - 서로 다른 검색 결과를 비교·보완하여 사례를 확장하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│  ID: 4cfb0d1e-c2d7-4c9f-a6cf-c489d32d62be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Multi Tool)                                                                           │
│                                                                                                                 │
│  Task: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER와 EXA semantic search를 조합하라.                                                                 │
│      - 서로 다른 검색 결과를 비교·보완하여 사례를 확장하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\USER\AppData\Local\Temp\ipykernel_30936\115422655.py:22: DeprecationWarning: search_and_contents() is deprecated. Use search() instead; search() returns text contents by default.
  result = exa.search_and_contents( # EXA API를 활용하여 의미 기반 검색을 수행하는 부분입니다. 사용자가 입력한 쿼리를 기반으로 EXA API를 통해 관련된 기업 사례를 검색하여 반환합니다.


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Multi Tool)                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 기업들의 AI 기반 데이터 분석 자동화 도입 사례                                                              │
│                                                                                                                 │
│  | 번호 | 기업명         | 산업               | 도입 배경                                              | 실행   │
│  방식                                                   | 성과/효과                                             │
│  | 출처 URL                                                                                     |               │
│  | ---- | -------------- | ------------------ | ------------------------------------------------------ |        │
│  ---------------------------------------------------------- |                                                   │
│  -------------------------------------------------------- |                                                     │
│  -------------------------------------------------------------------------------------------- |                 │
│  | 1    | 롯데백화점     | 소매업             | 방대한 고객 데이터를 효과적으로 분석하고 활용 필요  | AI 기반   │
│  BI 플랫폼 '스트래티지 원'을 도입해 고객 관계 분석 시스템 구축 | 분석 소요 시간 70% 단축, 신뢰성 높은 분석      │
│  결과 도출   | [ZDNet](https://zdnet.co.kr/view/?no=20250701105743)                                       |     │
│  | 2    | GS리테일       | 유통               | 대규모 고객 데이터 운영의 효율성 제고 필요            | AIOps   │
│  Agent 시스템 구축하여 인시던트 자율 분석 수행       | 인시던트 분석 시간 93% 단축                              │
│  | [AWS 기술 블로그](https://aws.amazon.com/ko/blogs/tech/gsretail-aiops-sre-agent/)           |                │
│  | 3    | 우리은행       | 금융               | 고객 응대 및 내부 통제 과정에서 효율성 향상 필요     | 175개의  │
│  AI 에이전트를 구축하여 전체 업무 자동화 추진      | 의사 결정 속도 향상, 리스크 관리 능력 강화            |    │
│  [이투데이](https://www.etoday.co.kr/news/view/2573399)                                    |                    │
│  | 4    | CJ올리브영     | 리테일             | 비개발직군도 AI 도구를 활용한 데이터 분석 필요        | 구글    │
│  클라우드 '제미나이 엔터프라이즈' 도입               | 결제추정금액 19% 증가, 고객 데이터 분석 효율성 개선   |  │
│  [디지털타임스](https://www.dt.co.kr/article/12057287)                                      |                   │
│  | 5    | 삼성SDS        | IT 서비스          | 대규모 AI 에이전트 구축으로 액셀러레이티드 자동화 필요 | AI     │
│  Agent를 통한 자동화 플랫폼 설계 및 실행                | 고객 요구사항 처리 시간 대폭 단축                     │
│  | [이투데이](https://www.etoday.co.kr/news/view/2573399)                                    |                  │
│                                                                                                                 │
│  ### 전체 비교 요약                                                                                             │
│  최근 3년간 AI 기반 데이터 분석 자동화 도입이 증가하고 있으며, 주로 유통 및 금융 산업에서 성과를 거두고 있다.   │
│  기업들은 AI 시스템을 도입해 분석 시간 단축과 비용 절감, 의사결정 효율성을 높이며 수익 증대에 기여하고 있다.    │
│  각 기업은 서로 다른 AI 기술과 시스템을 활용하였으나, 공통적으로 데이터 기반의 통찰력 확보와 운영 효율향상이    │
│  중요하게 다루어지고 있다.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [접근 방식]                                                                                                │
│      - SERPER와 EXA semantic search를 조합하라.                                                                 │
│      - 서로 다른 검색 결과를 비교·보완하여 사례를 확장하라.                                                     │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│  Agent: Research Analyst (Multi Tool)                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8a83149f-564a-4478-a654-a212cb5fac34                                                                       │
│  Final Output: ### 기업들의 AI 기반 데이터 분석 자동화 도입 사례                                                │
│                                                                                                                 │
│  | 번호 | 기업명         | 산업               | 도입 배경                                              | 실행   │
│  방식                                                   | 성과/효과                                             │
│  | 출처 URL                                                                                     |               │
│  | ---- | -------------- | ------------------ | ------------------------------------------------------ |        │
│  ---------------------------------------------------------- |                                                   │
│  -------------------------------------------------------- |                                                     │
│  -------------------------------------------------------------------------------------------- |                 │
│  | 1    | 롯데백화점     | 소매업             | 방대한 고객 데이터를 효과적으로 분석하고 활용 필요  | AI 기반   │
│  BI 플랫폼 '스트래티지 원'을 도입해 고객 관계 분석 시스템 구축 | 분석 소요 시간 70% 단축, 신뢰성 높은 분석      │
│  결과 도출   | [ZDNet](https://zdnet.co.kr/view/?no=20250701105743)                                       |     │
│  | 2    | GS리테일       | 유통               | 대규모 고객 데이터 운영의 효율성 제고 필요            | AIOps   │
│  Agent 시스템 구축하여 인시던트 자율 분석 수행       | 인시던트 분석 시간 93% 단축                              │
│  | [AWS 기술 블로그](https://aws.amazon.com/ko/blogs/tech/gsretail-aiops-sre-agent/)           |                │
│  | 3    | 우리은행       | 금융               | 고객 응대 및 내부 통제 과정에서 효율성 향상 필요     | 175개의  │
│  AI 에이전트를 구축하여 전체 업무 자동화 추진      | 의사 결정 속도 향상, 리스크 관리 능력 강화            |    │
│  [이투데이](https://www.etoday.co.kr/news/view/2573399)                                    |                    │
│  | 4    | CJ올리브영     | 리테일             | 비개발직군도 AI 도구를 활용한 데이터 분석 필요        | 구글    │
│  클라우드 '제미나이 엔터프라이즈' 도입               | 결제추정금액 19% 증가, 고객 데이터 분석 효율성 개선   |  │
│  [디지털타임스](https://www.dt.co.kr/article/12057287)                                      |                   │
│  | 5    | 삼성SDS        | IT 서비스          | 대규모 AI 에이전트 구축으로 액셀러레이티드 자동화 필요 | AI     │
│  Agent를 통한 자동화 플랫폼 설계 및 실행                | 고객 요구사항 처리 시간 대폭 단축                     │
│  | [이투데이](https://www.etoday.co.kr/news/view/2573399)                                    |                  │
│                                                                                                                 │
│  ### 전체 비교 요약                                                                                             │
│  최근 3년간 AI 기반 데이터 분석 자동화 도입이 증가하고 있으며, 주로 유통 및 금융 산업에서 성과를 거두고 있다.   │
│  기업들은 AI 시스템을 도입해 분석 시간 단축과 비용 절감, 의사결정 효율성을 높이며 수익 증대에 기여하고 있다.    │
│  각 기업은 서로 다른 AI 기술과 시스템을 활용하였으나, 공통적으로 데이터 기반의 통찰력 확보와 운영 효율향상이    │
│  중요하게 다루어지고 있다.                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== Tool 조합 결과 =====

### 기업들의 AI 기반 데이터 분석 자동화 도입 사례

| 번호 | 기업명         | 산업               | 도입 배경                                              | 실행 방식                                                   | 성과/효과                                                | 출처 URL                                                                                     |
| ---- | -------------- | ------------------ | ------------------------------------------------------ | ---------------------------------------------------------- | -------------------------------------------------------- | -------------------------------------------------------------------------------------------- |
| 1    | 롯데백화점     | 소매업             | 방대한 고객 데이터를 효과적으로 분석하고 활용 필요  | AI 기반 BI 플랫폼 '스트래티지 원'을 도입해 고객 관계 분석 시스템 구축 | 분석 소요 시간 70% 단축, 신뢰성 높은 분석 결과 도출   | [ZDNet](https://zdnet.co.kr/view/?no=20250701105743)                                       |
| 2    | GS리테일       | 유통               | 대규모 고객 데이터 운영의 효율성 제고 필요           

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 🔟 Crew 구성 (Planning-first)

In [28]:
crew_planning = Crew(
    agents=[planning_agent],
    tasks=[planning_task],
    verbose=True
)

### 🔟-B Crew 실행과 최종 결과 출력 (Planning-first)


In [29]:
planning_result = crew_planning.kickoff()

print("\n===== Planning-first 결과 =====\n")
print(planning_result)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.11.0                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 639a2022-33f5-45c3-954f-a06f96b3174e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [중요]                                                                                                     │
│      아래 구조를 반드시 따르라.                                                                                 │
│                                                                                                                 │
│      1) Plan                                                                                                    │
│      - 조사에 필요한 작업 단계를 먼저 3~5줄로 계획하라.                                                         │
│      - 어떤 Tool을 어느 단계에서 사용할지 간단히 밝히라.                                                        │
│                                                                                                                 │
│      2) Execute                                                                                                 │
│      - 계획에 따라 실제 사례를 조사하라.                                                                        │
│      - 최종 사례 정리는 아래 출력 형식을 따르라.                                                                │
│                                                                                                                 │
│      3) Reflect                                                                                                 │
│      - 결과를 스스로 점검하라.                                                                                  │
│      - 사례 중복, 출처 부족, 국내/해외 균형 부족 여부를 3~5줄로 검토하라.                                       │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│  ID: 6437eeb9-c6aa-4e7e-bc27-6af8391e3905                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: create_reasoning_plan                                                                                    │
│  Args: {'plan': 'Investigate and compare cases of AI-based data analysis automation implemented by companies    │
│  in the last three years, following a structured approach that includes necessary research tools.',...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planning-first Research Strategist                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### AI 기반 데이터 분석 자동화 도입 사례                                                                       │
│                                                                                                                 │
│  1) **롯데백화점**                                                                                              │
│     - **산업**: 유통                                                                                            │
│     - **도입 배경**: 고객관계관리(CRM) 분석 업무의 효율성을 높이고 데이터 기반 의사결정 체계를 강화하기 위해    │
│  생성형 AI 기반 BI 플랫폼 '스트래티지 원'을 도입.                                                               │
│     - **실행 방식**: 자연어 질의응답 기반으로 사용자가 질문을 하여 복잡한 분석 결과를 실시간으로 도출할 수      │
│  있는 AI 어시스턴트 '오토 봇'을 도입.                                                                           │
│     - **성과/효과**: 분석 시간이 몇 시간에서 몇 십 초로 단축되어 의사결정 속도가 획기적으로 개선됨.             │
│     - **출처 URL**: [ZDNet korea](https://zdnet.co.kr/view/?no=20250701105743)                                  │
│                                                                                                                 │
│  2) **현대모비스**                                                                                              │
│     - **산업**: 자동차 부품                                                                                     │
│     - **도입 배경**: 원가계산서 처리의 효율성을 높이기 위해 AI 기반 문서 처리 시스템 도입.                      │
│     - **실행 방식**: OCR과 KIE 기술을 활용하여 비정형 데이터를 정형 데이터로 전환하고 AI 에이전트를 통한 문서   │
│  AI 인프라 구축.                                                                                                │
│     - **성과/효과**: 업무 소요 시간이 대폭 단축되고, 보다 정확한 원가 계산 가능.                                │
│     - **출처 URL**: [한국딥러닝 블로그](https://www.koreadeep.com/blog/cost-calculation-automation)             │
│                                                                                                                 │
│  3) **우리은행 (SK C&C)**                                                                                       │
│     - **산업**: 금융                                                                                            │
│     - **도입 배경**: 다양한 비정형 데이터 자산화 및 효율적 활용을 위해 AI 시스템 구축.                          │
│     - **실행 방식**: 생성형 AI 모델을 적용한 ‘비정형 데이터 자산화 시스템’과 ‘AI 지식 상담 시스템’ 구축.        │
│     - **성과/효과**: 은행 업무의 자동화와 직원들의 업무 활용도를 높이고, 신뢰성을 개선함.                       │
│     - **출처 URL**: [연합뉴스](https://www.yna.co.kr/view/AKR20230926035800017)                                 │
│                                                                                                                 │
│  4) **농심**                                                                                                    │
│     - **산업**: 식품                                                                                            │
│     - **도입 배경**: 원자재 가격 예측과 체계적인 구매 전략 수립을 위해 AI 및 MLOps 아키텍처 도입.               │
│     - **실행 방식**: AWS SageMaker를 활용하여 데이터 수집 모듈과 머신러닝 모델을 자동화.                        │
│     - **성과/효과**: 데이터 기반 의사결정의 체계적 진행 및 과정에서의 인력 소모와 시간을 절약.                  │
│     - **출처 URL**: [Amazon                                                                                     │
│  AWS](https://aws.amazon.com/ko/blogs/tech/nongshim-amazon-sagemaker-material-forecast-mlops/)                  │
│                                                                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: [미션]                                                                                                   │
│      최근 3년 이내 기업들의 AI 기반 데이터 분석 자동화 도입 사례를 조사하고 비교하라                            │
│                                                                                                                 │
│      [중요]                                                                                                     │
│      아래 구조를 반드시 따르라.                                                                                 │
│                                                                                                                 │
│      1) Plan                                                                                                    │
│      - 조사에 필요한 작업 단계를 먼저 3~5줄로 계획하라.                                                         │
│      - 어떤 Tool을 어느 단계에서 사용할지 간단히 밝히라.                                                        │
│                                                                                                                 │
│      2) Execute                                                                                                 │
│      - 계획에 따라 실제 사례를 조사하라.                                                                        │
│      - 최종 사례 정리는 아래 출력 형식을 따르라.                                                                │
│                                                                                                                 │
│      3) Reflect                                                                                                 │
│      - 결과를 스스로 점검하라.                                                                                  │
│      - 사례 중복, 출처 부족, 국내/해외 균형 부족 여부를 3~5줄로 검토하라.                                       │
│                                                                                                                 │
│      [출력 형식]                                                                                                │
│  - 총 4~6개 사례를 정리하라.                                                                                    │
│  - 국내 1개 이상 + 해외 2개 이상 포함하라.                                                                      │
│  - 아래 항목을 각 사례마다 포함하라.                                                                            │
│    1) 기업명                                                                                                    │
│    2) 산업                                                                                                      │
│    3) 도입 배경                                                                                                 │
│    4) 실행 방식                                                                                                 │
│    5) 성과/효과                                                                                                 │
│    6) 출처 URL                                                                                                  │
│  - 마지막에 전체 비교 요약 5줄 이내로 정리하라.                                                                 │
│  - 가능한 경우 표 또는 번호 목록 형태로 구조화하라.                                                             │
│                                                                                                                 │
│  Planning:                                                                                                      │
│  Investigate and compare cases of AI-based data analysis automation implemented by companies in the last three  │
│  years, following a structured approach that includes necessary research

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 639a2022-33f5-45c3-954f-a06f96b3174e                                                                       │
│  Final Output: ### AI 기반 데이터 분석 자동화 도입 사례                                                         │
│                                                                                                                 │
│  1) **롯데백화점**                                                                                              │
│     - **산업**: 유통                                                                                            │
│     - **도입 배경**: 고객관계관리(CRM) 분석 업무의 효율성을 높이고 데이터 기반 의사결정 체계를 강화하기 위해    │
│  생성형 AI 기반 BI 플랫폼 '스트래티지 원'을 도입.                                                               │
│     - **실행 방식**: 자연어 질의응답 기반으로 사용자가 질문을 하여 복잡한 분석 결과를 실시간으로 도출할 수      │
│  있는 AI 어시스턴트 '오토 봇'을 도입.                                                                           │
│     - **성과/효과**: 분석 시간이 몇 시간에서 몇 십 초로 단축되어 의사결정 속도가 획기적으로 개선됨.             │
│     - **출처 URL**: [ZDNet korea](https://zdnet.co.kr/view/?no=20250701105743)                                  │
│                                                                                                                 │
│  2) **현대모비스**                                                                                              │
│     - **산업**: 자동차 부품                                                                                     │
│     - **도입 배경**: 원가계산서 처리의 효율성을 높이기 위해 AI 기반 문서 처리 시스템 도입.                      │
│     - **실행 방식**: OCR과 KIE 기술을 활용하여 비정형 데이터를 정형 데이터로 전환하고 AI 에이전트를 통한 문서   │
│  AI 인프라 구축.                                                                                                │
│     - **성과/효과**: 업무 소요 시간이 대폭 단축되고, 보다 정확한 원가 계산 가능.                                │
│     - **출처 URL**: [한국딥러닝 블로그](https://www.koreadeep.com/blog/cost-calculation-automation)             │
│                                                                                                                 │
│  3) **우리은행 (SK C&C)**                                                                                       │
│     - **산업**: 금융                                                                                            │
│     - **도입 배경**: 다양한 비정형 데이터 자산화 및 효율적 활용을 위해 AI 시스템 구축.                          │
│     - **실행 방식**: 생성형 AI 모델을 적용한 ‘비정형 데이터 자산화 시스템’과 ‘AI 지식 상담 시스템’ 구축.        │
│     - **성과/효과**: 은행 업무의 자동화와 직원들의 업무 활용도를 높이고, 신뢰성을 개선함.                       │
│     - **출처 URL**: [연합뉴스](https://www.yna.co.kr/view/AKR20230926035800017)                                 │
│                                                                                                                 │
│  4) **농심**                                                                                                    │
│     - **산업**: 식품                                                                                            │
│     - **도입 배경**: 원자재 가격 예측과 체계적인 구매 전략 수립을 위해 AI 및 MLOps 아키텍처 도입.               │
│     - **실행 방식**: AWS SageMaker를 활용하여 데이터 수집 모듈과 머신러닝 모델을 자동화.                        │
│     - **성과/효과**: 데이터 기반 의사결정의 체계적 진행 및 과정에서의 인력 소모와 시간을 절약.                  │
│     - **출처 URL**: [Amazon                                                                                     │
│  AWS](https://aws.amazon.com/ko/blogs/tech/nongshim-amazon-sagemaker-material-forecast-mlops/)                  │
│                                                                                                       


===== Planning-first 결과 =====

### AI 기반 데이터 분석 자동화 도입 사례

1) **롯데백화점**
   - **산업**: 유통
   - **도입 배경**: 고객관계관리(CRM) 분석 업무의 효율성을 높이고 데이터 기반 의사결정 체계를 강화하기 위해 생성형 AI 기반 BI 플랫폼 '스트래티지 원'을 도입.
   - **실행 방식**: 자연어 질의응답 기반으로 사용자가 질문을 하여 복잡한 분석 결과를 실시간으로 도출할 수 있는 AI 어시스턴트 '오토 봇'을 도입.
   - **성과/효과**: 분석 시간이 몇 시간에서 몇 십 초로 단축되어 의사결정 속도가 획기적으로 개선됨. 
   - **출처 URL**: [ZDNet korea](https://zdnet.co.kr/view/?no=20250701105743)

2) **현대모비스**
   - **산업**: 자동차 부품
   - **도입 배경**: 원가계산서 처리의 효율성을 높이기 위해 AI 기반 문서 처리 시스템 도입.
   - **실행 방식**: OCR과 KIE 기술을 활용하여 비정형 데이터를 정형 데이터로 전환하고 AI 에이전트를 통한 문서 AI 인프라 구축.
   - **성과/효과**: 업무 소요 시간이 대폭 단축되고, 보다 정확한 원가 계산 가능.
   - **출처 URL**: [한국딥러닝 블로그](https://www.koreadeep.com/blog/cost-calculation-automation)

3) **우리은행 (SK C&C)**
   - **산업**: 금융
   - **도입 배경**: 다양한 비정형 데이터 자산화 및 효율적 활용을 위해 AI 시스템 구축.
   - **실행 방식**: 생성형 AI 모델을 적용한 ‘비정형 데이터 자산화 시스템’과 ‘AI 지식 상담 시스템’ 구축.
   - **성과/효과**: 은행 업무의 자동화와 직원들의 업무 활용도를 높이고, 신뢰성을 개선함.
   - **출처 URL**: [연합뉴스](https://www.yna.co.kr/vie

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 1️⃣1️⃣ 간단한 정량 비교

In [30]:
import re

def simple_stats(text):
    t = str(text)
    urls = re.findall(r'https?://\S+', t)
    lines = [ln for ln in t.splitlines() if ln.strip()]
    return {
        "총 문자 수": len(t),
        "총 라인 수": len(lines),
        "URL 개수": len(urls),
        "고유 URL 개수": len(set(urls)),
        "Plan 포함 여부": "Plan" in t or "PLAN" in t,
        "Reflect 포함 여부": "Reflect" in t or "REFLECT" in t,
    }

single_stats = simple_stats(single_result)
multi_stats = simple_stats(multi_result)
planning_stats = simple_stats(planning_result)

print("단일 Tool 통계:", single_stats)
print("Tool 조합 통계:", multi_stats)
print("Planning-first 통계:", planning_stats)


단일 Tool 통계: {'총 문자 수': 2574, '총 라인 수': 10, 'URL 개수': 6, '고유 URL 개수': 6, 'Plan 포함 여부': False, 'Reflect 포함 여부': False}
Tool 조합 통계: {'총 문자 수': 2116, '총 라인 수': 10, 'URL 개수': 5, '고유 URL 개수': 4, 'Plan 포함 여부': False, 'Reflect 포함 여부': False}
Planning-first 통계: {'총 문자 수': 1838, '총 라인 수': 33, 'URL 개수': 5, '고유 URL 개수': 5, 'Plan 포함 여부': False, 'Reflect 포함 여부': False}


### 1️⃣2️⃣ LLM as a Judge (세 접근 방식 비교 평가)

In [31]:
from openai import OpenAI

client = OpenAI()

evaluation_prompt = f"""
당신은 '기업 사례 조사 전략 비교' 품질을 평가하는 심사위원이다.
아래는 동일한 미션에 대해 세 가지 접근 방식이 생성한 결과이다.

중요:
이번 평가는 단순히 '정보가 많고 자세한가'만 보는 것이 아니다.
Agentic AI 설계 관점에서, 각 접근이 얼마나 구조적으로 문제를 해결했는지도 함께 평가하라.
특히 Planning-first 접근은 정보량뿐 아니라,
작업 흐름 설계, 실행 순서의 명확성, Tool 활용의 목적성까지 반영하여 평가하라.

[미션]
{mission}

[단일 Tool 결과]
{single_result}

[Tool 조합 결과]
{multi_result}

[Planning-first 결과]
{planning_result}

다음 기준으로 각각 1~5점으로 평가하고, 근거를 2~3문장으로 설명하라.

1) 정보 품질
- 사례의 구체성
- 출처의 신뢰성
- 산업 및 사례 다양성

2) 구조화 수준
- 읽기 쉬움
- 항목 정리 수준
- 비교 가능성

3) 실행 안정성
- 요구사항 충족 여부(국내/해외, 출처 포함, 중복 최소화)
- 결과의 일관성
- 누락 여부

4) 전략성
- Tool 사용의 목적성이 분명한가
- 조사 범위와 흐름이 구조적으로 설계되었는가
- 단순 나열이 아니라 단계적 접근이 드러나는가

평가 시 유의사항:
- Planning-first 접근은 단순 정보량이 조금 적더라도,
  조사 흐름이 명확하고 실행 전략이 잘 드러나면 전략성에서 높게 평가하라.
- Tool 조합 접근은 정보가 풍부하더라도,
  조사 흐름이나 전략이 불명확하면 전략성 점수를 과도하게 높이지 마라.
- 단일 Tool 접근은 결과가 좋아 보여도,
  구조적 조사 전략이 약하면 전략성 점수는 제한하라.
- 이번 평가는 '누가 더 많은 정보를 모았는가'보다
  '누가 더 적절한 방식으로 문제를 해결했는가'를 함께 본다.

출력 형식:
1. 접근 방식별로 4개 항목 점수와 근거를 제시하라.
2. 각 접근의 총점을 20점 만점으로 제시하라.
3. 마지막에 '이번 미션에서 가장 적절한 접근'을 1개 선언하라.
4. 승자 선언 시 반드시 이번 실습 결과를 기준으로 구체적으로 설명하라.
5. 마크다운 형식으로 정리하라.
""".strip()

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a rigorous evaluator. Respond in Korean. Be fair, concise, and evidence-based."},
        {"role": "user", "content": evaluation_prompt},
    ],
    temperature=0.2,
)

print("\n===== LLM 평가 결과 =====\n")
print(resp.choices[0].message.content)



===== LLM 평가 결과 =====

# 기업 사례 조사 전략 비교 평가

## 1. 접근 방식별 평가

### 단일 Tool 결과
1) **정보 품질**: 4점  
   - 사례가 구체적이며, 출처도 신뢰할 수 있는 자료들로 구성되어 있다. 그러나 산업 및 사례의 다양성이 다소 제한적이다.

2) **구조화 수준**: 3점  
   - 표 형식으로 정리되어 있어 읽기 쉬우나, 항목 간 비교 가능성이 부족하다. 각 사례 간의 연관성을 명확히 드러내지 못했다.

3) **실행 안정성**: 4점  
   - 요구사항을 충족하며, 결과의 일관성이 높다. 다만, 중복 사례가 일부 존재한다.

4) **전략성**: 2점  
   - Tool 사용의 목적성이 다소 불명확하고, 조사 흐름이 구조적으로 설계되지 않았다. 단순 나열에 그치는 경향이 있다.

**총점**: 13점

---

### Tool 조합 결과
1) **정보 품질**: 5점  
   - 다양한 산업의 사례를 포함하고 있으며, 출처도 신뢰할 수 있는 자료들로 구성되어 있다. 정보량이 풍부하다.

2) **구조화 수준**: 4점  
   - 표 형식으로 잘 정리되어 있어 읽기 쉽고, 항목 간 비교가 가능하다. 그러나 일부 항목에서 더 명확한 정리가 필요하다.

3) **실행 안정성**: 4점  
   - 요구사항을 충족하며, 결과의 일관성이 높다. 다만, 중복 사례가 일부 존재한다.

4) **전략성**: 3점  
   - Tool 사용의 목적성이 다소 명확하나, 조사 흐름이 불명확한 부분이 있다. 단순 나열에 그치는 경향이 보인다.

**총점**: 16점

---

### Planning-first 결과
1) **정보 품질**: 4점  
   - 사례가 구체적이며, 출처도 신뢰할 수 있다. 그러나 정보량이 상대적으로 적다.

2) **구조화 수준**: 5점  
   - 각 사례가 명확하게 정리되어 있어 읽기 쉽고, 비교 가능성이 높다. 항목 간의 연관성이 잘 드러난다.

3) **실행 안정성**: 5점

### 1️⃣3️⃣ 실습 해석 가이드

- **단일 Tool 접근**: 빠르지만 정보의 깊이와 구조가 제한될 수 있습니다.
- **Tool 조합 접근**: 더 다양한 정보원을 확보할 수 있어 사례 확장에 유리합니다.
- **Planning-first 접근**: 실행 전에 구조를 잡기 때문에 결과의 일관성과 안정성이 높아지는 경향이 있습니다.

> 체크 포인트  
> 1. 요구사항(국내/해외, 출처 URL 등)을 가장 안정적으로 충족한 방식은 무엇인가?  
> 2. 가장 읽기 쉬운 결과를 만든 방식은 무엇인가?  
> 3. 앞으로 실무 프로젝트에서는 어떤 문제에 어떤 전략을 적용할 수 있을까?
